# NBA 賭盤勝率預測模型與市場效率分析

**研究問題：** FiveThirtyEight Elo 模型的預測勝率能有效預測 NBA 比賽結果嗎？加入背靠背賽程等特徵能否進一步提升準確率？

**資料來源：**
- Basketball-Reference.com（比賽結果）
- FiveThirtyEight NBA Elo Dataset（預測勝率）

**資料範圍：** 2005–2015 NBA 常規賽，共 13,283 場

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'Microsoft JhengHei'
matplotlib.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
print(f'資料筆數：{len(df)}')
print(f'賽季範圍：{df["season"].min()} – {df["season"].max()}')
df.head()

## 1. 資料概覽

In [ ]:
print('=== 基本統計 ===')
print(f'主場勝率：{df["home_win"].mean():.1%}')
print(f'Elo 預測準確率（勝率 > 0.5 視為預測主場勝）：{((df["home_win_prob"] > 0.5) == df["home_win"]).mean():.1%}')
print(f'Elo 預測勝率平均：{df["home_win_prob"].mean():.3f}')
print(f'\nElo 分布：\n{df["home_win_prob"].describe().round(3)}')

## 2. Elo 模型校準分析（預測勝率 vs 實際勝率）

In [ ]:
bins = np.arange(0.3, 0.85, 0.05)
df['prob_bin'] = pd.cut(df['home_win_prob'], bins=bins)
grouped = df.groupby('prob_bin')['home_win'].agg(['mean', 'count']).reset_index()
grouped['bin_center'] = [iv.mid for iv in grouped['prob_bin']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(grouped['bin_center'], grouped['mean'],
           s=grouped['count'] * 0.4, alpha=0.8, color='steelblue', label='實際勝率')
ax.plot([0.3, 0.8], [0.3, 0.8], 'r--', label='完美校準線')
ax.set_xlabel('Elo 預測勝率', fontsize=13)
ax.set_ylabel('實際勝率', fontsize=13)
ax.set_title('Elo 模型校準圖\n（點大小 = 樣本數）', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('→ 結論：Elo 預測勝率與實際勝率高度吻合，說明模型具備良好校準性。')

## 3. 主場優勢分析

In [ ]:
home_wr = df['home_win'].mean()
fav_wr  = df[df['home_win_prob'] > 0.5]['home_win'].mean()
dog_wr  = df[df['home_win_prob'] < 0.5]['home_win'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(['主場', '客場'], [home_wr, 1 - home_wr], color=['steelblue', 'salmon'])
axes[0].set_title('整體主客場勝率', fontsize=13)
for i, v in enumerate([home_wr, 1 - home_wr]):
    axes[0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 0.75)

axes[1].bar(['Elo 看好主場\n（主場贏率）', 'Elo 看好客場\n（冷門主場贏率）'],
            [fav_wr, dog_wr], color=['steelblue', 'orange'])
axes[1].set_title('Elo 預測方向 vs 主場實際勝率', fontsize=13)
for i, v in enumerate([fav_wr, dog_wr]):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 0.85)

plt.suptitle('主場優勢分析', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()
print(f'→ 結論：NBA 主場平均勝率 {home_wr:.1%}，Elo 看好主場時勝率達 {fav_wr:.1%}。')

## 4. 背靠背賽程效應

In [ ]:
b2b_groups = {
    '雙方正常': df[(df['home_b2b']==0) & (df['away_b2b']==0)]['home_win'].mean(),
    '主場背靠背': df[(df['home_b2b']==1) & (df['away_b2b']==0)]['home_win'].mean(),
    '客場背靠背': df[(df['home_b2b']==0) & (df['away_b2b']==1)]['home_win'].mean(),
    '雙方背靠背': df[(df['home_b2b']==1) & (df['away_b2b']==1)]['home_win'].mean(),
}
b2b_counts = {
    '雙方正常': len(df[(df['home_b2b']==0) & (df['away_b2b']==0)]),
    '主場背靠背': len(df[(df['home_b2b']==1) & (df['away_b2b']==0)]),
    '客場背靠背': len(df[(df['home_b2b']==0) & (df['away_b2b']==1)]),
    '雙方背靠背': len(df[(df['home_b2b']==1) & (df['away_b2b']==1)]),
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(b2b_groups.keys(), b2b_groups.values(),
              color=['steelblue','salmon','seagreen','gray'], alpha=0.85)
ax.axhline(df['home_win'].mean(), color='black', linestyle='--',
           alpha=0.5, label=f'整體平均 {df["home_win"].mean():.1%}')
ax.set_ylabel('主場勝率', fontsize=12)
ax.set_title('背靠背賽程對主場勝率的影響', fontsize=14)
ax.set_ylim(0.45, 0.75); ax.legend()
for bar, (k, v) in zip(bars, b2b_groups.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.005,
            f'{v:.1%}\n(n={b2b_counts[k]})', ha='center', fontsize=9.5)
plt.tight_layout(); plt.show()
delta = b2b_groups['客場背靠背'] - b2b_groups['主場背靠背']
print(f'→ 結論：客場背靠背使主場勝率提升 {delta:.1%}，背靠背賽程是顯著影響因素。')

## 5. 逐賽季趨勢

In [ ]:
season_stats = df.groupby('season').apply(lambda s: pd.Series({
    'home_wr':    s['home_win'].mean(),
    'elo_acc':    ((s['home_win_prob'] > 0.5) == s['home_win']).mean(),
    'upset_rate': s[s['home_win_prob'] < 0.5]['home_win'].mean(),
})).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
titles = ['主場勝率', 'Elo 預測準確率', '冷門發生率（Elo 看客場但主場贏）']
cols   = ['home_wr', 'elo_acc', 'upset_rate']
colors = ['steelblue', 'seagreen', 'salmon']

for ax, col, title, color in zip(axes, cols, titles, colors):
    ax.plot(season_stats['season'], season_stats[col], 'o-', color=color, linewidth=2)
    ax.axhline(season_stats[col].mean(), color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel(title, fontsize=11); ax.grid(True, alpha=0.3)

axes[0].set_title('逐賽季趨勢分析（2005–2015）', fontsize=14)
axes[2].set_xlabel('賽季', fontsize=12)
plt.tight_layout(); plt.show()
print('→ 結論：主場優勢在 2014-2015 賽季有下降趨勢，與 NBA 引入背靠背賽程優化政策時間吻合。')

## 6. 預測模型比較

In [ ]:
features_adv = ['home_win_prob', 'elo_diff', 'home_b2b', 'away_b2b', 'b2b_advantage', 'month']
df_clean = df.dropna(subset=features_adv + ['home_win'])
X_base = df_clean[['home_win_prob']]
X_adv  = df_clean[features_adv]
y      = df_clean['home_win']
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {
    '基準線\n（永遠猜主場）': y.mean(),
    'Elo 勝率\n（單一特徵）': cross_val_score(LogisticRegression(), X_base, y, cv=cv).mean(),
    'Elo + 背靠背\n（邏輯迴歸）': cross_val_score(LogisticRegression(), X_adv, y, cv=cv).mean(),
    '隨機森林\n（進階特徵）': cross_val_score(RandomForestClassifier(100, random_state=42), X_adv, y, cv=cv).mean(),
    '梯度提升\n（進階特徵）': cross_val_score(GradientBoostingClassifier(random_state=42), X_adv, y, cv=cv).mean(),
}

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['gray','steelblue','royalblue','seagreen','darkorange']
bars = ax.bar(results.keys(), results.values(), color=colors)
ax.set_ylabel('5-Fold 交叉驗證準確率', fontsize=12)
ax.set_title('各預測模型準確率比較', fontsize=14)
ax.set_ylim(0.55, 0.73)
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

best = max(results, key=results.get)
print(f'→ 最佳模型：{best.replace(chr(10)," ")}，準確率 {results[best]:.1%}')

## 9. 完整研究結論

### Elo 模型分析（2005–2015，13,283 場）

| 發現 | 數據 |
|------|------|
| NBA 主場優勢 | 平均勝率 **59.6%** |
| Elo 模型基礎準確率 | **67.4%**（遠高於基準線 59.6%）|
| 背靠背效應 | 客場背靠背時主場勝率提升 **+4.2%**（59.1% → 63.3%）|
| 最佳預測模型 | 邏輯迴歸（Elo + 背靠背），準確率 **67.3%** |

### 運彩賠率 × 市場效率分析（2008–2015，9,025 場）

| 發現 | 數據 |
|------|------|
| 賠率 Brier Score | **0.1957**（Elo 為 0.2026，賠率更準） |
| 莊家平均抽水率 | **3.70%**，每注期望虧損 1.85% |
| 背靠背賠率定價誤差 | 客場背靠背僅差 **-0.3%**（市場充分定價）|
| 市場效率結論 | 賠率已整合 Elo 以外資訊，NBA 市場效率相對高 |

### 主要發現
1. **Elo 模型具備良好校準性**：預測勝率與實際勝率高度吻合，但賠率校準性略優（Brier Score 低 3.4%）。
2. **NBA 賠率市場高效率**：背靠背等公開資訊已充分定價（差距 < 0.5%），難以創造套利空間。
3. **莊家 Overround 穩定**：平均 3.70%，各賽季差異小，反映成熟市場定價能力。
4. **主場優勢逐年下降**：2014-2015 賽季降至 57.5%，可能與聯盟排程政策調整有關。
5. **加入特徵對預測提升有限**：Elo 勝率已高度概括球隊實力，背靠背等附加特徵貢獻邊際。

### 未來方向
- 加入球員傷病與陣容資料（如 Basketball-Reference 每場 lineup）
- 分析開盤賠率 vs 收盤賠率差異（line movement）作為市場情緒指標
- 分析個別球隊的主場優勢差異與主客場勝率的統計顯著性

In [ ]:
groups_b2b = {
    '雙方正常':   df_odds[(df_odds['home_b2b']==0) & (df_odds['away_b2b']==0)],
    '主場背靠背': df_odds[(df_odds['home_b2b']==1) & (df_odds['away_b2b']==0)],
    '客場背靠背': df_odds[(df_odds['home_b2b']==0) & (df_odds['away_b2b']==1)],
    '雙方背靠背': df_odds[(df_odds['home_b2b']==1) & (df_odds['away_b2b']==1)],
}

labels   = list(groups_b2b.keys())
actual   = [g['home_win'].mean()       for g in groups_b2b.values()]
mkt_prob = [g['home_fair_prob'].mean() for g in groups_b2b.values()]
elo_prob = [g['home_win_prob'].mean()  for g in groups_b2b.values()]
counts   = [len(g) for g in groups_b2b.values()]

x = np.arange(len(labels))
width = 0.28

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width, actual,   width, label='實際主場勝率', color='steelblue', alpha=0.85)
b2 = ax.bar(x,         mkt_prob, width, label='賠率隱含勝率', color='crimson',   alpha=0.85)
b3 = ax.bar(x + width, elo_prob, width, label='Elo 預測勝率', color='seagreen',  alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('主場勝率', fontsize=12)
ax.set_title('背靠背賽程：實際勝率 vs 賠率定價 vs Elo 預測', fontsize=14)
ax.legend(fontsize=11); ax.set_ylim(0.45, 0.80); ax.grid(True, alpha=0.3, axis='y')

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.1%}', ha='center', fontsize=9)
for i, n in enumerate(counts):
    ax.text(x[i], 0.462, f'n={n}', ha='center', fontsize=8.5, color='gray')

plt.tight_layout(); plt.show()

away_b2b = groups_b2b['客場背靠背']
mkt_gap  = away_b2b['home_win'].mean() - away_b2b['home_fair_prob'].mean()
elo_gap  = away_b2b['home_win'].mean() - away_b2b['home_win_prob'].mean()
print(f'客場背靠背時：')
print(f'  實際主場勝率：{away_b2b["home_win"].mean():.1%}')
print(f'  賠率隱含勝率：{away_b2b["home_fair_prob"].mean():.1%}  （差距 {mkt_gap:+.1%}）')
print(f'  Elo 預測勝率：{away_b2b["home_win_prob"].mean():.1%}   （差距 {elo_gap:+.1%}）')
print()
print('→ 結論：賠率幾乎完整反映背靠背效應（差距 < 0.5%），Elo 稍微低估。')
print('   背靠背已是市場充分定價的公開資訊，無法作為套利基礎。')

### 8.3 背靠背賽程：賠率定價充分嗎？

In [ ]:
vig_pct = (df_odds['overround'] - 1) * 100
season_vig = df_odds.groupby('season').apply(
    lambda s: (s['overround'].mean() - 1) * 100
).reset_index(name='avg_vig')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(vig_pct.dropna(), bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(vig_pct.mean(), color='crimson', linestyle='--', linewidth=2,
                label=f'平均 {vig_pct.mean():.2f}%')
axes[0].set_xlabel('Overround（%）', fontsize=12)
axes[0].set_ylabel('場數', fontsize=12)
axes[0].set_title('莊家抽水率分布', fontsize=14)
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

axes[1].plot(season_vig['season'], season_vig['avg_vig'], 'o-',
             color='steelblue', linewidth=2, markersize=8)
axes[1].axhline(season_vig['avg_vig'].mean(), color='gray', linestyle='--', alpha=0.5,
                label=f'總平均 {season_vig["avg_vig"].mean():.2f}%')
axes[1].set_xlabel('賽季', fontsize=12)
axes[1].set_ylabel('平均 Overround（%）', fontsize=12)
axes[1].set_title('莊家抽水率逐賽季趨勢', fontsize=14)
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.suptitle('莊家抽水率（Overround）分析', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

print(f'→ 結論：平均 Overround = {vig_pct.mean():.2f}%，即每注期望虧損 {vig_pct.mean()/2:.2f}%。')
print(f'   長期盲目下注理論上每百元虧損約 {vig_pct.mean()/2:.2f} 元，市場對賭徒不利。')

### 8.2 莊家抽水率（Overround / Vig）分析

In [ ]:
bins = np.arange(0.30, 0.85, 0.05)

def get_calibration(df, prob_col):
    df = df.copy()
    df['_bin'] = pd.cut(df[prob_col], bins=bins)
    g = df.groupby('_bin')['home_win'].agg(['mean','count']).reset_index()
    g['center'] = [iv.mid for iv in g['_bin']]
    return g

g_mkt = get_calibration(df_odds, 'home_fair_prob')
g_elo = get_calibration(df_odds, 'home_win_prob')

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0.3, 0.8], [0.3, 0.8], 'k--', alpha=0.4, label='完美校準線', linewidth=1.5)
ax.scatter(g_mkt['center'], g_mkt['mean'],
           s=g_mkt['count'] * 0.5, color='crimson', alpha=0.85, zorder=3,
           label='運彩賠率隱含勝率（去vig）')
ax.plot(g_mkt['center'], g_mkt['mean'], '-', color='crimson', alpha=0.4)
ax.scatter(g_elo['center'], g_elo['mean'],
           s=g_elo['count'] * 0.5, color='steelblue', alpha=0.85, zorder=3,
           label='FiveThirtyEight Elo 預測')
ax.plot(g_elo['center'], g_elo['mean'], '-', color='steelblue', alpha=0.4)
ax.set_xlabel('預測勝率', fontsize=13)
ax.set_ylabel('實際勝率', fontsize=13)
ax.set_title('校準比較：運彩賠率 vs Elo 模型\n（點大小 = 樣本數，2008–2015）', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

brier_mkt = ((df_odds['home_fair_prob'] - df_odds['home_win']) ** 2).mean()
brier_elo = ((df_odds['home_win_prob']  - df_odds['home_win']) ** 2).mean()
print(f'Brier Score（越低越準確）：')
print(f'  運彩賠率：{brier_mkt:.4f}')
print(f'  Elo 模型：{brier_elo:.4f}')
print(f'  差異：{(brier_elo - brier_mkt):.4f}（賠率優於 Elo {(brier_elo/brier_mkt - 1):.1%}）')
print()
print('→ 結論：賠率 Brier Score 略優於 Elo，市場整合了 Elo 以外的資訊（如傷病、交易等），但差距不大。')

### 8.1 校準比較：賠率 vs Elo（Brier Score）

In [ ]:
df_odds = pd.read_csv('../data/processed/odds_features.csv', parse_dates=['date'])
df_odds = df_odds.dropna(subset=['home_fair_prob', 'home_win_prob', 'home_win'])

print(f'資料筆數：{len(df_odds)}')
print(f'賽季範圍：{df_odds["season"].min()} – {df_odds["season"].max()}')
print(f'平均莊家 Overround：{(df_odds["overround"].mean() - 1):.2%}  →  每注期望虧損 {(df_odds["overround"].mean() - 1)/2:.2%}')
print(f'\n賠率公平勝率平均：{df_odds["home_fair_prob"].mean():.3f}')
print(f'Elo 預測勝率平均：{df_odds["home_win_prob"].mean():.3f}')
df_odds[['home_fair_prob','home_win_prob','overround','prob_diff']].describe().round(4)

---
## 8. 運彩賠率 vs Elo 模型：市場效率分析

**資料來源：** sportsbookreviewsonline.com（Pinnacle/市場平均 Moneyline 賠率）  
**資料範圍：** 2007–08 至 2014–15 賽季，共 9,025 場（與 Elo 資料交叉匹配）

**核心問題：** 莊家賠率是否比 FiveThirtyEight Elo 更準確預測比賽結果？背靠背等因素是否已被市場充分定價？